# Day 12: Subgraphs and Multi-Agent Systems

As your graphs grow, two problems show up.

First, one big graph with 30 nodes is hard to read, hard to test, and hard to reuse. The fix is the same as in normal programming: break it into smaller pieces. In LangGraph, **a whole graph can be used as a single node inside another graph.** That's a subgraph.

Second, one agent with 20 tools gets confused. It picks the wrong tool, forgets instructions, and the prompt gets enormous. The fix is the same as in a company: you don't hire one person to do sales, accounts and IT. You hire specialists, and someone coordinates them. That's a **multi-agent system**.

By the end of today you'll be able to:

- use a compiled graph as a node in another graph
- call a subgraph that has completely different state keys
- see what's happening inside a subgraph while streaming
- build a team of agents with a **supervisor** who decides who works next
- build agents that **hand off** to each other directly

In [ ]:
%pip install -q -U langgraph langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app, xray=False):
    # xray=True also draws what's inside any subgraphs
    try:
        display(Image(app.get_graph(xray=xray).draw_mermaid_png()))
    except Exception:
        print(app.get_graph(xray=xray).draw_mermaid())

## 1. A graph inside a graph

Think of a car factory. The engine is built on its own assembly line, tested on its own, and then dropped into the car as a single part. The main car line doesn't care how the engine was built; it just installs it.

When the subgraph and the parent graph use the **same state keys**, this is as easy as passing the compiled subgraph to `add_node`.

```
Parent:   START -> get_input -> [ clean ] -> report -> END
Inside "clean":    START -> strip_spaces -> lower -> END
```

First we build the small graph on its own.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class TextState(TypedDict):
    text: str
    report: str

In [ ]:
def strip_spaces(state: TextState) -> dict:
    return {"text": " ".join(state["text"].split())}

In [ ]:
def lower(state: TextState) -> dict:
    return {"text": state["text"].lower()}

In [ ]:
sub_builder = StateGraph(TextState)

In [ ]:
sub_builder.add_sequence([strip_spaces, lower])

In [ ]:
sub_builder.add_edge(START, "strip_spaces")
sub_builder.add_edge("lower", END)

In [ ]:
clean_subgraph = sub_builder.compile()

In [ ]:
# It's a normal graph, so we can test it on its own first
print(clean_subgraph.invoke({"text": "  HELLO    World  "}))

Now the parent graph. Look at the `add_node` line: we pass the whole compiled graph as if it were a function.

In [ ]:
def get_input(state: TextState) -> dict:
    return {"text": state["text"] + "   !!!"}

In [ ]:
def report(state: TextState) -> dict:
    return {"report": f"Cleaned text: '{state['text']}'"}

In [ ]:
parent_builder = StateGraph(TextState)

In [ ]:
parent_builder.add_node("get_input", get_input)
parent_builder.add_node("clean", clean_subgraph)      # a whole graph as one node
parent_builder.add_node("report", report)

In [ ]:
parent_builder.add_edge(START, "get_input")
parent_builder.add_edge("get_input", "clean")
parent_builder.add_edge("clean", "report")
parent_builder.add_edge("report", END)

In [ ]:
parent_app = parent_builder.compile()
show_graph(parent_app, xray=True)

In [ ]:
print(parent_app.invoke({"text": "   HELLO    LangGraph  "})["report"])

## 2. A subgraph with its own private state

Very often the subgraph was written for a different purpose and uses **different keys**. Then you can't just plug it in. Instead, call it from inside a normal node, and translate between the two:

1. take what you need from the parent's state, and shape it as the subgraph's input
2. call `subgraph.invoke(...)`
3. take what you need from the subgraph's output, and return it as the parent's update

It's like using a translator at a meeting. You tell the translator what you want to say, they talk to the other side in their language, and they tell you the reply in yours.

Here's a reusable "translate" subgraph that knows nothing about questions and answers.

In [ ]:
class TranslateState(TypedDict):
    source: str
    target_language: str
    translated: str

In [ ]:
def translate(state: TranslateState) -> dict:
    prompt = f"Translate into {state['target_language']}. Reply with only the translation: {state['source']}"
    return {"translated": llm.invoke(prompt).content}

In [ ]:
translate_builder = StateGraph(TranslateState)

In [ ]:
translate_builder.add_node("translate", translate)

In [ ]:
translate_builder.add_edge(START, "translate")
translate_builder.add_edge("translate", END)

In [ ]:
translator = translate_builder.compile()

And a parent graph that only knows about `question`, `answer` and `answer_hindi`.

In [ ]:
class QAState(TypedDict):
    question: str
    answer: str
    answer_hindi: str

In [ ]:
def answer(state: QAState) -> dict:
    return {"answer": llm.invoke(f"Answer in one sentence: {state['question']}").content}

In [ ]:
def to_hindi(state: QAState) -> dict:
    # 1. shape the subgraph's input from the parent's state
    subgraph_input = {"source": state["answer"], "target_language": "Hindi"}

    # 2. run the subgraph
    subgraph_output = translator.invoke(subgraph_input)

    # 3. hand back only what the parent cares about
    return {"answer_hindi": subgraph_output["translated"]}

In [ ]:
qa_builder = StateGraph(QAState)

In [ ]:
qa_builder.add_sequence([answer, to_hindi])

In [ ]:
qa_builder.add_edge(START, "answer")
qa_builder.add_edge("to_hindi", END)

In [ ]:
qa_app = qa_builder.compile()

In [ ]:
result = qa_app.invoke({"question": "What is the capital of Maharashtra?"})
print("English:", result["answer"])
print("Hindi:  ", result["answer_hindi"])

## 3. Seeing inside a subgraph while streaming

By default, streaming shows a subgraph as one step, like a closed box. Add `subgraphs=True` to see the steps inside it too. Each item then comes as a pair: `(namespace, update)`. The namespace is empty for the parent, and names the subgraph otherwise.

### Predict before you run

The parent has 3 nodes, and the `clean` subgraph has 2 nodes inside. With `subgraphs=True`, how many updates will be printed?

<details>
<summary>Click to see the answer</summary>

Five. Three from the parent (`get_input`, `clean`, `report`) and two from inside the subgraph (`strip_spaces`, `lower`). The two inner ones appear before the parent's `clean` update, because the subgraph has to finish before `clean` counts as done.
</details>

In [ ]:
for namespace, update in parent_app.stream({"text": "  ABC  DEF "}, stream_mode="updates", subgraphs=True):
    if namespace:
        where = "inside " + namespace[0].split(":")[0]
    else:
        where = "parent"
    print(f"{where:15} {update}")

## 4. A team of agents with a supervisor

Now the second big idea. Instead of one agent that does everything, we build a small team:

- a **researcher**, an agent with a search tool (we'll use a tiny made-up knowledge base)
- a **writer**, who turns the research into a friendly answer
- a **supervisor**, who looks at the conversation and decides who should work next, or whether we're done

Think of a project manager in a stand-up meeting. They don't write the code or the documentation. They look at where things are and say "Researcher, you're up", then "Writer, your turn", then "Great, we're done."

```
                   +---------> researcher ---+
  START -> supervisor <----------------------+
                   +---------> writer -------+
                   |
                   +--> FINISH -> END
```

Each worker reports back to the supervisor when it's done, using `Command(goto="supervisor")` from Day 3.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, SystemMessage
from langchain.agents import create_agent
from langgraph.graph import MessagesState
from langgraph.types import Command

In [ ]:
NOTES = {
    "langgraph": "LangGraph is a library for building stateful, multi-step LLM agents as graphs. It was released by LangChain in 2024.",
    "checkpointer": "A checkpointer saves the graph's state after every step. This enables memory, human-in-the-loop and time travel.",
    "send": "Send lets a graph run one node many times in parallel, each with different input. This is used for map-reduce.",
}

In [ ]:
@tool
def search_notes(query: str) -> str:
    """Search our internal notes. Use simple keywords like 'langgraph', 'checkpointer' or 'send'."""
    matches = [text for keyword, text in NOTES.items() if keyword in query.lower()]
    if not matches:
        return "No notes found."
    return "\n".join(matches)

In [ ]:
research_agent = create_agent(
    model=llm,
    tools=[search_notes],
    system_prompt="You are a researcher. Search the notes and report the facts you found. Be brief.",
)

Our team's state is a normal chat state, plus a step counter. Multi-agent loops can go round and round, so the counter is our safety net (Day 3 again).

In [ ]:
class TeamState(MessagesState):
    steps: int

In [ ]:
def researcher(state: TeamState) -> Command[Literal["supervisor"]]:
    result = research_agent.invoke({"messages": state["messages"]})

    # Label the message with who wrote it, so the supervisor can tell
    note = AIMessage(content=result["messages"][-1].content, name="researcher")
    return Command(update={"messages": [note]}, goto="supervisor")

In [ ]:
WRITER_PROMPT = SystemMessage(content=(
    "You are a writer. Using the research in the conversation, write a clear, "
    "friendly answer for a beginner in at most 80 words."
))

In [ ]:
def writer(state: TeamState) -> Command[Literal["supervisor"]]:
    reply = llm.invoke([WRITER_PROMPT] + state["messages"])
    answer = AIMessage(content=reply.content, name="writer")
    return Command(update={"messages": [answer]}, goto="supervisor")

Now the supervisor. It uses structured output (Day 10) so its decision is always one of three exact words.

In [ ]:
class NextWorker(BaseModel):
    next: Literal["researcher", "writer", "FINISH"] = Field(
        description="researcher if facts are still missing, writer if facts exist but there's no final answer yet, FINISH once the writer has answered"
    )

In [ ]:
supervisor_llm = llm.with_structured_output(NextWorker)

In [ ]:
def supervisor(state: TeamState) -> Command[Literal["researcher", "writer", "__end__"]]:
    steps = state.get("steps", 0)
    if steps >= 4:
        return Command(goto=END)           # safety limit, stop no matter what

    # Show the supervisor a short version of who said what so far
    lines = []
    for message in state["messages"]:
        speaker = getattr(message, "name", None) or message.type
        lines.append(f"{speaker}: {message.content[:200]}")
    conversation = "\n".join(lines)

    decision = supervisor_llm.invoke(
        f"You manage a researcher and a writer.\nConversation so far:\n{conversation}\n\nWho should act next?"
    )
    print("Supervisor decided:", decision.next)

    if decision.next == "FINISH":
        return Command(update={"steps": steps + 1}, goto=END)
    return Command(update={"steps": steps + 1}, goto=decision.next)

In [ ]:
builder = StateGraph(TeamState)

In [ ]:
builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)

In [ ]:
builder.add_edge(START, "supervisor")     # every other route comes from Command

In [ ]:
team = builder.compile()
show_graph(team)

In [ ]:
result = team.invoke({"messages": [("user", "Explain what a checkpointer is in LangGraph.")], "steps": 0})

print()
for message in result["messages"]:
    speaker = getattr(message, "name", None) or message.type
    print(f"[{speaker}] {message.content}\n")

## 5. Agents that hand off to each other

A supervisor is one way to coordinate. Another is the way a call centre transfers you: the first person you talk to either helps you, or says "let me transfer you to the right department". No manager in the middle.

Each agent here returns either an answer, or a `Command(goto="the_other_agent")`. To stop two agents from passing the customer back and forth forever, we allow only one transfer: after that, whoever has the customer must answer.

```
  START -> sales <--> support -> END
```

In [ ]:
class AgentReply(BaseModel):
    reply: str = Field(description="your answer, or empty if you are handing off")
    handoff: Literal["none", "sales", "support"] = Field(description="who should handle this. 'none' if you answered it yourself")

In [ ]:
class DeskState(MessagesState):
    handoffs: int

In [ ]:
def make_desk_agent(name: str, role: str):
    brain = llm.with_structured_output(AgentReply)

    def desk_agent(state: DeskState) -> Command[Literal["sales", "support", "__end__"]]:
        already_transferred = state.get("handoffs", 0) >= 1

        if already_transferred:
            rule = "The customer was already transferred to you, so you MUST answer (handoff='none')."
        else:
            rule = "If the question isn't about your area, hand off instead of answering."

        instructions = SystemMessage(content=f"You are the {name} agent. {role} {rule}")
        decision = brain.invoke([instructions] + state["messages"])

        wants_to_transfer = decision.handoff not in ("none", name)
        if wants_to_transfer and not already_transferred:
            print(f"{name} is transferring the customer to {decision.handoff}")
            return Command(update={"handoffs": 1}, goto=decision.handoff)

        reply = AIMessage(content=decision.reply, name=name)
        return Command(update={"messages": [reply]}, goto=END)

    return desk_agent

In [ ]:
builder = StateGraph(DeskState)

In [ ]:
builder.add_node("sales", make_desk_agent("sales", "You handle prices, plans and discounts."))
builder.add_node("support", make_desk_agent("support", "You handle bugs, login problems and technical help."))

In [ ]:
builder.add_edge(START, "sales")       # every customer starts with sales

In [ ]:
help_desk = builder.compile()
show_graph(help_desk)

In [ ]:
for question in ["How much is the premium plan?", "I can't log in, it shows error 403"]:
    result = help_desk.invoke({"messages": [("user", question)], "handoffs": 0})
    last = result["messages"][-1]
    print("Customer:", question)
    print(f"[{last.name}]", last.content)
    print()

So which one should you use?

- **Supervisor**: one place makes every decision. Easier to control and debug. A good default.
- **Handoffs**: each agent decides for itself. Feels natural for customer-service style flows.
- **Hierarchical**: supervisors managing other supervisors, each team built as a subgraph. For very large systems.

## 6. Common mistakes

**Mistake 1: No step limit in a multi-agent loop.**
Two LLMs can easily get stuck: the supervisor keeps sending work to the researcher, who keeps finding the same thing. Every round costs money. Always keep a counter like `steps` and stop at a sensible number.

**Mistake 2: Forgetting to label messages.**
If every agent's message is just `AIMessage(content=...)`, the supervisor can't tell who said what. Use `name="researcher"` and so on.

**Mistake 3: Plugging in a subgraph with different keys directly.**
If the subgraph's keys don't match the parent's, `add_node("x", subgraph)` won't pass the data you expect. Call it inside a node and translate, like `to_hindi` in section 2.

**Mistake 4: Leaving out the `Literal` on a `Command` node.**
`def researcher(state) -> Command:` works, but LangGraph then can't draw where it goes, and your diagram shows disconnected boxes. Write `Command[Literal["supervisor"]]`.

**Mistake 5: Using the tiny model for a complicated supervisor.**
`gpt-4.1-nano` is great for simple steps, but coordinating a team is harder. If your supervisor makes odd choices, try `OPENAI_MODEL=gpt-4.1-mini` in your `.env` before rewriting your prompts.

## 7. Practice

**Exercise 1.** Make a subgraph `score_text` (keys: `text`, `score`) that sets `score` to the number of words. Use it as a node in a parent graph with the same keys plus `label`, where `label` is `"long"` if the score is more than 5, otherwise `"short"`.

**Exercise 2.** Add a third specialist to the supervisor team: a **fact checker** that says whether the writer's answer matches the research. You'll need a new node, a new option in the supervisor's choices, and a new supervisor function.

In [ ]:
# Solution 1
class ScoreState(TypedDict):
    text: str
    score: int
    label: str

In [ ]:
def count_words(state: ScoreState) -> dict:
    return {"score": len(state["text"].split())}

In [ ]:
score_builder = StateGraph(ScoreState)

In [ ]:
score_builder.add_node("count_words", count_words)

In [ ]:
score_builder.add_edge(START, "count_words")
score_builder.add_edge("count_words", END)

In [ ]:
score_text = score_builder.compile()

In [ ]:
def add_label(state: ScoreState) -> dict:
    if state["score"] > 5:
        return {"label": "long"}
    return {"label": "short"}

In [ ]:
label_builder = StateGraph(ScoreState)

In [ ]:
label_builder.add_node("score_text", score_text)       # the subgraph as a node
label_builder.add_node("add_label", add_label)

In [ ]:
label_builder.add_edge(START, "score_text")
label_builder.add_edge("score_text", "add_label")
label_builder.add_edge("add_label", END)

In [ ]:
label_app = label_builder.compile()

In [ ]:
result = label_app.invoke({"text": "one two three four five six seven"})
print(result)
assert result["label"] == "long"

In [ ]:
# Solution 2
CHECKER_PROMPT = SystemMessage(content=(
    "Check whether the writer's answer is supported by the researcher's notes. "
    "Reply 'Verified', or describe the problem in one sentence."
))

In [ ]:
def fact_checker(state: TeamState) -> Command[Literal["supervisor"]]:
    verdict = llm.invoke([CHECKER_PROMPT] + state["messages"])
    message = AIMessage(content=verdict.content, name="fact_checker")
    return Command(update={"messages": [message]}, goto="supervisor")

In [ ]:
class NextOfThree(BaseModel):
    next: Literal["researcher", "writer", "fact_checker", "FINISH"] = Field(
        description="the usual order is researcher, then writer, then fact_checker, then FINISH"
    )

In [ ]:
three_way_llm = llm.with_structured_output(NextOfThree)

In [ ]:
def supervisor_of_three(state: TeamState) -> Command[Literal["researcher", "writer", "fact_checker", "__end__"]]:
    steps = state.get("steps", 0)
    if steps >= 5:
        return Command(goto=END)

    lines = []
    for message in state["messages"]:
        speaker = getattr(message, "name", None) or message.type
        lines.append(f"{speaker}: {message.content[:200]}")

    decision = three_way_llm.invoke(
        "Team: researcher, writer, fact_checker.\n" + "\n".join(lines) + "\n\nWho should act next?"
    )
    print("Supervisor decided:", decision.next)

    if decision.next == "FINISH":
        return Command(update={"steps": steps + 1}, goto=END)
    return Command(update={"steps": steps + 1}, goto=decision.next)

In [ ]:
builder = StateGraph(TeamState)

In [ ]:
builder.add_node("supervisor", supervisor_of_three)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)
builder.add_node("fact_checker", fact_checker)

In [ ]:
builder.add_edge(START, "supervisor")

In [ ]:
team_of_three = builder.compile()

In [ ]:
result = team_of_three.invoke({"messages": [("user", "What does Send do in LangGraph?")], "steps": 0})
print(result["messages"][-1].content)

## Wrapping up

A compiled graph can be a **node** in another graph when the keys match, or be **called inside a node** when they don't (translate in, run, translate out). `subgraphs=True` lets you see inside while streaming, and `xray=True` draws the inside.

For teams of agents, a **supervisor** picks the next specialist and the specialists report back, while **handoffs** let agents transfer work directly with `Command(goto=...)`. Either way, label your messages and always keep a step limit.

Tomorrow we connect agents to **your own documents**. That's RAG, probably the most requested feature in real company projects.